In [ ]:
import pandas as pd     

In [ ]:
customers = pd.read_csv("../data/customers.csv")
history = pd.read_csv("../data/customer_history.csv")
ref_train = pd.read_csv("../data/referance_data.csv")
ref_test = pd.read_csv("../data/referance_data_test.csv")

In [ ]:
for name, df in zip(
    ["customers", "history", "ref_train", "ref_test"],
    [customers, history, ref_train, ref_test]
):
    print(f"{name}: {df.shape}")
    display(df.head())

In [ ]:
history["date"] = pd.to_datetime(history["date"])
ref_train["ref_date"] = pd.to_datetime(ref_train["ref_date"])
ref_test["ref_date"] = pd.to_datetime(ref_test["ref_date"])

In [ ]:
print(customers.isnull().sum())
print(history.isnull().sum())
print(ref_train.isnull().sum())
print(ref_test.isnull().sum())

In [ ]:
print(customers.dtypes)
print(history.dtypes)

In [ ]:
len(customers["cust_id"].unique()), len(history["cust_id"].unique())
# müsteri sayısı

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.histplot(customers["age"], bins=30, kde=True)
plt.title("Müşteri Yaş Dağılımı")
plt.xlabel("Yaş")
plt.ylabel("Frekans")
plt.show()

sns.boxplot(x="gender",y="tenure", data=customers)
plt.title("Cinsiyete Göre Müşterililik Süresi")
plt.xlabel("Cinsiyet")
plt.ylabel("Müşterililik Süresi (tenure)")
plt.show()

In [ ]:
sample_ids = history["cust_id"].drop_duplicates().sample(2, random_state=42)
for cid in sample_ids:
    subset = history[history["cust_id"] == cid].sort_values("date")
    plt.plot(subset["date"], subset["cc_transaction_all_amt"], marker='o', label=f'Cust ID: {cid}')
plt.legend()
plt.title("Örnek Müşterilerin Kredi Kartı İşlem Tutarları Zaman Serisi")
plt.xlabel("Tarih")
plt.ylabel("Kredi Kartı İşlem Tutarı")
plt.show()

In [ ]:
# Customer.csv için
print(customers.isna().sum().sort_values(ascending=False))
print(customers.info())

In [ ]:
# Customer.csv için

# Kategorik değişkenlerdeki eksik değerleri "Unknown" ile dolduralım
cat_cols = ["gender", "province", "religion", "work_type","work_sector"]
for col in cat_cols:
    customers[col] = customers[col].fillna("Unknown")

# Numerik değişkenlerdeki eksik değerleri medyan ile dolduralım
num_cols = ["age", "tenure"]
for col in num_cols:
    customers[col] = customers[col].fillna(customers[col].median())


In [ ]:
# Customer_history.csv için
print(history.isna().sum().sort_values(ascending=False))
print(history.info())

In [ ]:
# Customer_history.csv için

# Numerik değişkenlerdeki eksik değerleri 0 ile dolduralım
numeric_cols = ["cc_transaction_all_amt", "cc_transaction_all_cnt", "mobile_eft_all_amt", "mobile_eft_all_cnt","active_product_category_nbr"]

for col in numeric_cols:
    history[col] = history[col].fillna(0)


In [ ]:
import numpy as np

# Aykırı değer eşikleri
def outlier_threshold(df, col, q1=0.25, q3=0.75):
    q1_val = df[col].quantile(q1)
    q3_val = df[col].quantile(q3)
    IQR = q3_val - q1_val
    lower = q1_val - 1.5 * IQR
    upper = q3_val + 1.5 * IQR
    return lower, upper

# Aykırı değer var mı?
def check_outlier(df, col):
    lower, upper = outlier_threshold(df, col)
    return (df[col] < lower).any() or (df[col] > upper).any()

# Aykırı değerleri eşiklerle değiştir
def replace_with_thresholds(df, col):
    lower, upper = outlier_threshold(df, col)
    df.loc[df[col] < lower, col] = lower
    df.loc[df[col] > upper, col] = upper

# Uygulama örneği
num_cols = [
    "mobile_eft_all_cnt",
    "mobile_eft_all_amt",
    "active_product_category_nbr",
    "cc_transaction_all_amt",
    "cc_transaction_all_cnt"
]


for col in num_cols:
    if check_outlier(history, col):
        replace_with_thresholds(history, col)

# Customers tarafı
# Yaşı mantıksal sınırla kesiyoruz
customers["age"] = customers["age"].clip(lower=18, upper=100)

In [ ]:
def grab_col_names(dataframe, cat_th=10, car_th=20):
    """
    Veri setindeki değişken türlerini otomatik olarak sınıflandırır.

    Parameters
    ----------
    dataframe: pd.DataFrame
        İncelenecek veri seti
    cat_th: int
        Numerik ama az sayıda unique değeri olan değişkenleri kategorik olarak işaretleme eşiği
    car_th: int
        Kategorik ama çok fazla sınıfı (cardinality) olan değişkenleri ayırma eşiği

    Returns
    -------
    cat_cols: list
        Kategorik değişkenler
    num_cols: list
        Numerik değişkenler
    cat_but_car: list
        Kategorik görünümlü ama yüksek kardinal değişkenler
    """

    # 1. Kategorik kolonlar
    cat_cols = [col for col in dataframe.columns if dataframe[col].dtype == "O" or str(dataframe[col].dtype) in ["category", "bool"]]

    # 2. Numerik görünümlü ama az sınıflı kolonlar (örnek: 0-1 flag)
    num_but_cat = [col for col in dataframe.columns 
                   if dataframe[col].nunique() < cat_th and dataframe[col].dtype.kind in ['i', 'f']]

    # 3. Kategorik görünümlü ama çok fazla sınıflı (örnek: ID, postal_code)
    cat_but_car = [col for col in cat_cols if dataframe[col].nunique() > car_th]

    # 4. Güncellenmiş kategorik kolonlar listesi
    cat_cols = cat_cols + num_but_cat
    cat_cols = [col for col in cat_cols if col not in cat_but_car]

    # 5. Numerik kolonlar
    num_cols = [col for col in dataframe.columns if dataframe[col].dtype.kind in ['i', 'f']]
    num_cols = [col for col in num_cols if col not in num_but_cat]

    # 6. Bilgi çıktısı
    print(f"Observation: {dataframe.shape[0]}")
    print(f"Variables: {dataframe.shape[1]}")
    print(f"Cat cols: {len(cat_cols)}")
    print(f"Num cols: {len(num_cols)}")
    print(f"Cat but car: {len(cat_but_car)}")
    print(f"Num but cat: {len(num_but_cat)}")

    return cat_cols, num_cols, cat_but_car


In [ ]:
print("CUSTOMERS")
cat_cols_cust, num_cols_cust, cat_but_car_cust = grab_col_names(customers)

print("\nHISTORY")
cat_cols_hist, num_cols_hist, cat_but_car_hist = grab_col_names(history)


In [ ]:
# ==========================================================
# CLEANED DATASETS - SAVE
# ==========================================================

# Temiz versiyonları kaydediyoruz
customers.to_csv("../data/customers_clean.csv", index=False)
history.to_csv("../data/history_clean.csv", index=False)

print("✅ Temizlenmiş veriler başarıyla kaydedildi!")


In [ ]:
customers[num_cols_cust].corr()

# Matrisi
f,ax = plt.subplots(figsize=(18,13))
sns.heatmap(customers[num_cols_cust].corr(),annot=True,cmap="YlGnBu")
ax.set_title("Correlation Heatmap")
plt.show()


history[num_cols_hist].corr()
# Matrisi
f,ax = plt.subplots(figsize=(18,13))
sns.heatmap(history[num_cols_hist].corr(),annot=True,cmap="YlGnBu")
ax.set_title("Correlation Heatmap")
plt.show()
